In [121]:
!uv pip install numpy pandas yfinance matplotlib seaborn plotly TA-Lib

Using Python 3.13.5 environment at: /home/michabay/projects/trading-bot/.venv
Audited 7 packages in 4ms


In [122]:
from datetime import datetime
from zoneinfo import ZoneInfo

import yfinance as yf
import pandas as pd
import numpy as np
import talib

In [123]:
SYMBOLS: list[str] = ["AMZN", "TGT", "WMT", "NKE", "HP"]
# Get the stock data
YF_DFS = yf.download(
    SYMBOLS, group_by="Ticker", start="2013-01-01",
    end="2025-08-18", auto_adjust=True
)

[*********************100%***********************]  5 of 5 completed


In [124]:
def rearrange_df(odf: pd.DataFrame) -> pd.DataFrame:
    df = odf.reset_index()
    df.rename(columns={
        "Date": "timestamp",
        "Open": "open",
        "High": "high",
        "Low": "low",
        "Close": "close",
        "Volume": "volume",
    }, inplace=True)

    zi = ZoneInfo("US/Eastern")
    df["timestamp"] = df["timestamp"].apply(
        lambda x: datetime.fromisoformat(str(x)).astimezone(zi)
    )

    return df.set_index("timestamp")

assert YF_DFS is not None

DATA: dict = {}
for symbol in SYMBOLS:
    df = YF_DFS[symbol]
    assert isinstance(df, pd.DataFrame)
    DATA[symbol] = rearrange_df(df)


In [125]:
def chunk_data(
    df: pd.DataFrame, n_groups: int = 15, split_pct: float = 0.7,
    min_chunk_size: int = 150
) -> list[tuple[int, int, int]]:
    assert 0.2 < split_pct < 0.8, "Split should be between 0.2 and 0.8."

    chunk_size: int = len(df.index) // n_groups
    assert chunk_size >= min_chunk_size, f"min_size: {min_chunk_size} < chunk_size: {chunk_size}"

    i: int = 0
    chunks: list[tuple[int, int, int]] = []
    while i < len(df) - chunk_size:
        old_i: int = i
        i += chunk_size
        split_i = int(old_i + (i - old_i) * split_pct)
        if len(df) - i < min_chunk_size:
            i = len(df)
        
        chunks.append((old_i, split_i, i))
    
    return chunks

CHUNKS = {symbol: chunk_data(DATA[symbol]) for symbol in SYMBOLS}

In [ ]:
def mean_revert(close: pd.Series, period: int = 25, threshold: int = 2) -> None:
    returns = np.log(close / close.shift(1))
    # Compute SMA with ma_period
    sma_values = talib.SMA(close.to_numpy(), timeperiod=period)

    distance = close - sma_values

    # When trend line is greater than upper threshold --> short
    position = np.where(distance > threshold, -1, np.nan)

    # When trend is less than lower threshold --> long
    position = np.where(distance < -threshold, 1, position)

    # When trend cross back into threshold area --> close position (go neutral)
    position = np.where(
        distance * distance.shift(1) < 0, 0, position
    )
    # Fill NA's with 0
    # position = position.fillna(0)
    # strategy = position.shift(1) * returns


symbol: str = "AMZN"
cs: list[tuple[int, int, int]] = CHUNKS[symbol]
df: pd.DataFrame = DATA[symbol]
mean_revert(df["close"])
# for chunk in cs:
#     train = df["close"][chunk[0]:chunk[1]]
#     test = df["close"][chunk[1]:chunk[2]]


3175
[nan nan nan ... nan  0. -1.]
